# generator-project-and-reshape — faded example 3: Fill the doubling transposed-conv

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `generator-project-and-reshape`. The last cell reports your progress on the `GAN: Generator project + reshape` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: Generator project + reshape` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`generator-project-and-reshape`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "generator-project-and-reshape"
DD_SUBTOPIC = "GAN: Generator project + reshape"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A `ConvTranspose2d(kernel_size=4, stride=2, padding=1)` exactly doubles the spatial size: `(in-1)*2 - 2 + 4 = 2*in`. Two of them take a 4x4 seed to 16x16. Getting the (k=4, s=2, p=1) triple right is what produces the clean 2x upsample.

## Faded exercise 3

Implement `Generator416`. The projection and `up2` are given; define `up1` as a `ConvTranspose2d` that doubles spatial size from the seed channels `base_C` to `base_C//2`. Complete the blanked `up1` layer.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(5)

class Generator416(nn.Module):
    def __init__(self, latent_dim, base_C=64, out_C=3):
        super().__init__()
        self.base_C = base_C
        self.project = nn.Linear(latent_dim, base_C * 4 * 4)
        self.up1 = nn.ConvTranspose2d(base_C, base_C // 2, kernel_size=4, stride=2, padding=1)
        self.up2 = nn.ConvTranspose2d(base_C // 2, out_C, kernel_size=4, stride=2, padding=1)

    def forward(self, z):
        B = z.shape[0]
        seed = self.project(z).view(B, self.base_C, 4, 4)
        mid = self.up1(seed)
        out = self.up2(mid)
        return {'seed': seed, 'mid': mid, 'out': out}

print({k: tuple(v.shape) for k, v in Generator416(100, 64, 3)(t.randn(2, 100)).items()})


def _test():
    g = Generator416(latent_dim=100, base_C=64, out_C=3)
    res = g(t.randn(2, 100))
    # independently derived shapes from the (k=4,s=2,p=1) doubling rule
    assert tuple(res['seed'].shape) == (2, 64, 4, 4), res['seed'].shape
    assert tuple(res['mid'].shape) == (2, 32, 8, 8), res['mid'].shape
    assert tuple(res['out'].shape) == (2, 3, 16, 16), res['out'].shape
    # up1 must be a transposed conv with the doubling config
    assert isinstance(g.up1, nn.ConvTranspose2d)
    assert g.up1.stride == (2, 2) and g.up1.kernel_size == (4, 4) and g.up1.padding == (1, 1)


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn

t.manual_seed(5)

class Generator416(nn.Module):
    def __init__(self, latent_dim, base_C=64, out_C=3):
        super().__init__()
        self.base_C = base_C
        self.project = nn.Linear(latent_dim, base_C * 4 * 4)
        self.up1 = nn.ConvTranspose2d(base_C, base_C // 2, kernel_size=4, stride=2, padding=1)
        self.up2 = nn.ConvTranspose2d(base_C // 2, out_C, kernel_size=4, stride=2, padding=1)

    def forward(self, z):
        B = z.shape[0]
        seed = self.project(z).view(B, self.base_C, 4, 4)
        mid = self.up1(seed)
        out = self.up2(mid)
        return {'seed': seed, 'mid': mid, 'out': out}

print({k: tuple(v.shape) for k, v in Generator416(100, 64, 3)(t.randn(2, 100)).items()})
```
</details>